# Génération de légendes avec des VLMs quantifiés

*Projet 7 : VLMs légers pour légender ou juger des images.*

Préparation des images, chargement des deux VLMs en 16, 8 et 4 bits, génération des légendes et mesure de la mémoire et de la vitesse.

GPU requis dans Colab : *Exécution → Modifier le type d'exécution → T4 GPU*. Exécute les cellules dans l'ordre ; ce qui a déjà été calculé est repris depuis le Drive partagé.

## Mise en place (commune aux trois notebooks)

> **Dossier partagé** : Wafae crée `projet7_vlm` dans son Google Drive et le partage avec Meriem et Fatima (en éditrices). Meriem et Fatima l'ajoutent à *Mon Drive* : clic droit → *Organiser* → *Ajouter un raccourci à Drive*. Ainsi `MyDrive/projet7_vlm` pointe vers le même dossier pour les trois.
>
> **Ne pas modifier** `SEED`, `N_IMAGES`, `MAX_SIDE` sans accord de l'équipe : sinon les images ne sont plus les mêmes.

In [ ]:
!pip install -q -U "transformers>=4.49" accelerate bitsandbytes datasets pycocoevalcap gradio num2words scipy

In [ ]:
import os, gc, json, re, time, random, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

assert torch.cuda.is_available(), "Aucun GPU détecté : active un GPU (Exécution → Modifier le type d'exécution)."
print("GPU :", torch.cuda.get_device_name(0))
print("Mémoire GPU totale : %.1f Go" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

# bfloat16 seulement sur GPU Ampere ou plus récent (A100, L4...). Sur T4 → float16.
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("Type des poids en 16 bits :", DTYPE)

# ------------------------- CONFIGURATION -------------------------
SEED = 42
N_IMAGES = 300            # taille du sous-ensemble évalué (le sujet conseille 500 à 2000 ; 300 ≈ 1 h sur T4)
MAX_SIDE = 512            # redimensionnement des images (côté le plus long, en pixels)
MAX_NEW_TOKENS = 40       # longueur max d'une légende

MODELS = {
    "Qwen2-VL-2B": "Qwen/Qwen2-VL-2B-Instruct",
    "SmolVLM-2B":  "HuggingFaceTB/SmolVLM-Instruct",
}
PRECISIONS = ["16bit", "8bit", "4bit"]

CAPTION_PROMPT = "Describe this image in one short sentence."

# Option : sauvegarder sur Google Drive pour ne rien perdre en cas de déconnexion
USE_DRIVE = True   # dossier partagé de l'équipe : MyDrive/projet7_vlm
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/projet7_vlm")
else:
    ROOT = Path("/content/projet7_vlm") if Path("/content").exists() else Path("projet7_vlm")

DATA_DIR = ROOT / "data"
IMG_DIR = DATA_DIR / "images"
RES_DIR = ROOT / "results"
for d in (IMG_DIR, RES_DIR):
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Dossier de travail :", ROOT)

## 2. Préparation des données — Flickr8k

Flickr8k contient ~8 000 images, chacune avec **5 légendes humaines**. On prend un sous-ensemble aléatoire (graine fixe) de la partie *test*, on **redimensionne** les images (côté max = `MAX_SIDE`) et on les enregistre sur disque : **les mêmes images** serviront à toutes les configurations (16 / 8 / 4 bits), ce qui rend la comparaison équitable.

In [ ]:
from datasets import load_dataset

def prepare_image(img, max_side=MAX_SIDE):
    # Convertit en RGB et réduit l'image pour que son plus grand côté fasse max_side pixels.
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side), Image.BICUBIC)
    return img

META_PATH = DATA_DIR / "subset.json"

if META_PATH.exists():
    subset = json.loads(META_PATH.read_text())
    print(f"Sous-ensemble déjà préparé : {len(subset)} images")
else:
    ds = load_dataset("jxie/flickr8k", split="test")          # 1000 images, colonnes image + caption_0..4
    caption_cols = sorted(c for c in ds.column_names if c.startswith("caption"))
    idx = sorted(random.Random(SEED).sample(range(len(ds)), min(N_IMAGES, len(ds))))
    subset = []
    for i in tqdm(idx, desc="Préparation des images"):
        ex = ds[i]
        img_id = f"img_{i:04d}"
        prepare_image(ex["image"]).save(IMG_DIR / f"{img_id}.jpg", quality=95)
        subset.append({"id": img_id, "refs": [ex[c].strip() for c in caption_cols]})
    META_PATH.write_text(json.dumps(subset, indent=1, ensure_ascii=False))
    print(f"{len(subset)} images préparées dans {IMG_DIR}")

REFS = {s["id"]: s["refs"] for s in subset}
IDS = [s["id"] for s in subset]

def load_img(img_id):
    return Image.open(IMG_DIR / f"{img_id}.jpg").convert("RGB")

In [ ]:
# Aperçu de quelques images et de leurs légendes humaines
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, img_id in zip(axes, IDS[:4]):
    ax.imshow(load_img(img_id)); ax.axis("off")
    ax.set_title("\n".join(r[:45] for r in REFS[img_id][:2]), fontsize=8)
plt.tight_layout(); plt.show()

## 3. Chargement des VLMs en 16, 8 et 4 bits

La **quantification** stocke les poids du modèle avec moins de bits :

| Précision | Méthode | Mémoire des poids (≈ pour 2 B paramètres) |
|---|---|---|
| 16 bits | float16 / bfloat16 | ~4,4 Go |
| 8 bits | `bitsandbytes` LLM.int8 | ~2,5 Go |
| 4 bits | `bitsandbytes` NF4 + double quantification | ~1,5 Go |

On ne ré-entraîne pas les VLMs : on fait uniquement de **l'inférence**, comme le demande le sujet.

In [ ]:
import transformers
from packaging import version
from transformers import AutoProcessor, BitsAndBytesConfig
# Nouvelles versions de transformers : argument `dtype` ; anciennes : `torch_dtype`
DTYPE_KW = {"dtype": DTYPE} if version.parse(transformers.__version__) >= version.parse("4.56") else {"torch_dtype": DTYPE}
print("transformers", transformers.__version__)
try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoVLM

def quant_config(precision):
    if precision == "16bit":
        return None
    if precision == "8bit":
        return BitsAndBytesConfig(load_in_8bit=True)
    if precision == "4bit":
        return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                  bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    raise ValueError(precision)

def load_vlm(name, precision):
    model_id = MODELS[name]
    # On limite la résolution vue par le modèle pour garder un nombre de tokens image raisonnable
    if "Qwen" in model_id:
        proc_kwargs = dict(min_pixels=128 * 28 * 28, max_pixels=512 * 28 * 28)
    elif "SmolVLM" in model_id:
        proc_kwargs = dict(size={"longest_edge": 2 * 384})
    else:
        proc_kwargs = {}
    try:
        processor = AutoProcessor.from_pretrained(model_id, **proc_kwargs)
    except Exception:
        processor = AutoProcessor.from_pretrained(model_id)
    model = AutoVLM.from_pretrained(model_id, device_map="cuda:0",
                                    quantization_config=quant_config(precision), **DTYPE_KW)
    model.eval()
    return model, processor

def unload(*objs):
    for o in objs:
        del o
    gc.collect(); torch.cuda.empty_cache()

@torch.inference_mode()
def vlm_generate(model, processor, image, prompt, max_new_tokens=MAX_NEW_TOKENS):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=text, images=[image], return_tensors="pt")
    inputs = {k: (v.to(model.device, DTYPE) if torch.is_floating_point(v) else v.to(model.device))
              for k, v in inputs.items()}
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

def clean_caption(text):
    # Garde la première ligne / phrase et retire un éventuel préfixe « Assistant: ».
    text = re.sub(r"^\s*(assistant\s*:)\s*", "", text.strip(), flags=re.I)
    text = text.split("\n")[0].strip()
    m = re.match(r"(.{15,}?[.!?])(\s|$)", text)
    return (m.group(1) if m else text).strip()

In [ ]:
# Test rapide : un modèle en 4 bits sur une image
m, p = load_vlm("SmolVLM-2B", "4bit")
img = load_img(IDS[0])
print("VLM  :", clean_caption(vlm_generate(m, p, img, CAPTION_PROMPT)))
print("Réf. :", REFS[IDS[0]][0])
unload(m, p); del m, p; gc.collect(); torch.cuda.empty_cache()

### Premier test des trois précisions

In [ ]:
# Premier essai des 3 précisions sur le même modèle et les mêmes 3 images (résultats sauvegardés pour GitHub)
test_rows = []
for prec in PRECISIONS:
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    m, p = load_vlm("SmolVLM-2B", prec)
    mem = m.get_memory_footprint() / 1e9
    print(f"=== SmolVLM-2B {prec} — poids : {mem:.2f} Go")
    for img_id in IDS[:3]:
        t = time.time()
        cap = clean_caption(vlm_generate(m, p, load_img(img_id), CAPTION_PROMPT))
        dt = time.time() - t
        print(f"  {img_id} ({dt:.1f} s) : {cap}")
        test_rows.append({"precision": prec, "image": img_id, "poids_Go": round(mem, 2),
                          "temps_s": round(dt, 2), "legende": cap, "reference": REFS[img_id][0]})
    unload(m, p); del m, p; gc.collect(); torch.cuda.empty_cache()

test_df = pd.DataFrame(test_rows)
test_df.to_csv(RES_DIR / "test_precisions.csv", index=False)
shutil.copy(META_PATH, RES_DIR / "subset.json")      # liste des images choisies + légendes humaines
test_df

## 4. Génération des légendes et mesures

Pour chaque modèle × précision, on mesure :
- **mémoire des poids** (`get_memory_footprint`) et **pic de mémoire GPU** pendant la génération ;
- **temps de chargement** et **temps moyen par image** ;
- les **légendes** générées (sauvegardées en CSV).

Les configurations déjà calculées sont rechargées depuis `results/` (reprise après déconnexion).

In [ ]:
def run_config(name, precision, ids=IDS):
    tag = f"{name}__{precision}"
    cap_path, stat_path = RES_DIR / f"captions_{tag}.csv", RES_DIR / f"stats_{tag}.json"
    if cap_path.exists() and stat_path.exists():
        df = pd.read_csv(cap_path)
        if set(df["id"]) >= set(ids):
            print(f"[déjà calculé] {tag}")
            return
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    model, processor = load_vlm(name, precision)
    load_time = time.time() - t0
    footprint = model.get_memory_footprint() / 1e9

    vlm_generate(model, processor, load_img(ids[0]), CAPTION_PROMPT)   # échauffement (non chronométré)
    torch.cuda.reset_peak_memory_stats()

    rows, times = [], []
    for img_id in tqdm(ids, desc=tag):
        img = load_img(img_id)
        torch.cuda.synchronize(); t = time.time()
        raw = vlm_generate(model, processor, img, CAPTION_PROMPT)
        torch.cuda.synchronize(); times.append(time.time() - t)
        rows.append({"id": img_id, "raw": raw, "caption": clean_caption(raw)})

    stats = {"model": name, "precision": precision, "load_time_s": load_time,
             "weights_GB": footprint, "peak_GPU_GB": torch.cuda.max_memory_allocated() / 1e9,
             "time_per_image_s": float(np.mean(times)), "n_images": len(ids)}
    pd.DataFrame(rows).to_csv(cap_path, index=False)
    stat_path.write_text(json.dumps(stats, indent=1))
    print(stats)
    unload(model, processor); del model, processor
    gc.collect(); torch.cuda.empty_cache()

for name in MODELS:
    for prec in PRECISIONS:
        run_config(name, prec)

In [ ]:
# Tableau mémoire / vitesse
stats = pd.DataFrame([json.loads(p.read_text()) for p in sorted(RES_DIR.glob("stats_*.json"))])
stats.to_csv(RES_DIR / "couts_memoire_vitesse.csv", index=False)
stats.round(3)

## Coût de la quantification

In [ ]:
stats = pd.DataFrame([json.loads(p.read_text()) for p in sorted(RES_DIR.glob("stats_*.json"))])
models = list(MODELS)
colors = {"16bit": "#2a6fdb", "8bit": "#f2a541", "4bit": "#3bb273"}
cols = [("weights_GB", "Mémoire des poids (Go)"), ("peak_GPU_GB", "Pic mémoire GPU (Go)"),
        ("time_per_image_s", "Temps par image (s)"), ("load_time_s", "Temps de chargement (s)")]
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
x = np.arange(len(models)); w = 0.26
for ax, (col, title) in zip(axes, cols):
    for j, prec in enumerate(PRECISIONS):
        vals = [stats[(stats.model == m) & (stats.precision == prec)][col].values[0] for m in models]
        bars = ax.bar(x + (j - 1) * w, vals, w, label=prec, color=colors[prec])
        ax.bar_label(bars, fmt="%.2f", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(models); ax.set_title(title)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(title="Précision")
plt.tight_layout(); plt.savefig(RES_DIR / "couts_quantification.png", dpi=150); plt.show()

# Gain de mémoire relatif au 16 bits
for m in models:
    base = stats[(stats.model == m) & (stats.precision == "16bit")]["weights_GB"].values[0]
    for prec in PRECISIONS[1:]:
        v = stats[(stats.model == m) & (stats.precision == prec)]["weights_GB"].values[0]
        print(f"{m} {prec} : {v:.2f} Go, soit {100 * (1 - v / base):.0f} % de mémoire en moins que le 16 bits")

In [ ]:
# Exemples : la même image légendée par les 6 configurations
for img_id in IDS[:4]:
    plt.figure(figsize=(4, 4)); plt.imshow(load_img(img_id)); plt.axis("off"); plt.show()
    print("Référence humaine :", REFS[img_id][0])
    for name in MODELS:
        for prec in PRECISIONS:
            df = pd.read_csv(RES_DIR / f"captions_{name}__{prec}.csv").set_index("id")
            print(f"  {name:12s} {prec:6s} → {df.loc[img_id, 'caption']}")
    print("-" * 100)

### À analyser
- Gain de mémoire du 8 et du 4 bits par rapport au 16 bits.
- Effet sur la vitesse : le 8 bits est-il plus lent ? Pourquoi ?
- Les légendes changent-elles beaucoup d'une précision à l'autre ?

## Mettre à jour GitHub
1. *Fichier → Enregistrer une copie dans GitHub* : dépôt `vlm-compacts`, branche **ta branche** (wafae, meriem ou fatima), chemin **`notebooks/01_donnees_generation.ipynb`**, avec un message qui dit ce qui a été fait, par exemple `Ajout des graphiques de comparaison`.
2. Exécute la cellule ci-dessous, décompresse le zip et ajoute les fichiers dans le dossier `results/` de ta branche, puis ouvre une pull request vers `main` (*Add file → Upload files*).

In [ ]:
# Rassemble les résultats légers (CSV, JSON, PNG) à pousser — jamais les images ni les modèles
import zipfile, glob
zip_path = ROOT / "results_wafae.zip"
files = []
for pat in ['test_precisions.csv', 'subset.json', 'captions_*.csv', 'stats_*.json', 'couts_*']:
    files += glob.glob(str(RES_DIR / pat))
with zipfile.ZipFile(zip_path, "w") as z:
    for f in sorted(set(files)):
        z.write(f, arcname=Path(f).name)
print(len(set(files)), "fichiers →", zip_path)
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except Exception:
    pass